# 3. Research Agent with Model Context Protocol (MCP)

This notebook demonstrates how to connect research agents to external Model Context Protocol (MCP) tool servers.

In [ ]:
import os
import json
import asyncio
from typing import List, Dict, Any
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.tools import tool, BaseTool
from langchain_openai import ChatOpenAI

load_dotenv()

## 1. MCP Client Protocol Adapter

In [ ]:
class MCPClient:
    """Model Context Protocol client for accessing external knowledge servers."""
    def __init__(self, server_name: str):
        self.server_name = server_name
        
    async def list_tools(self) -> List[Dict[str, Any]]:
        return [
            {
                "name": "mcp_document_lookup",
                "description": "Lookup internal whitepapers and technical specs.",
                "schema": {"query": "str"}
            },
            {
                "name": "mcp_metrics_db",
                "description": "Query benchmark metrics and hardware specifications.",
                "schema": {"metric_name": "str"}
            }
        ]
    
    async def call_tool(self, tool_name: str, arguments: Dict[str, Any]) -> str:
        if tool_name == "mcp_document_lookup":
            return f"[MCP Verified Doc] Data for '{arguments.get('query')}': High-fidelity benchmarks confirm 99.9% 2-qubit gate fidelity in latest testbed."
        elif tool_name == "mcp_metrics_db":
            return f"[MCP Metrics DB] Query result: Coherence time T1 = 150 microseconds, T2 = 120 microseconds."
        return f"MCP tool {tool_name} returned default response."

## 2. Converting MCP Tools to LangChain Tools

In [ ]:
client = MCPClient("research_knowledge_server")

@tool
async def mcp_document_lookup(query: str) -> str:
    """Lookup technical documentation and whitepapers from verified MCP knowledge server."""
    return await client.call_tool("mcp_document_lookup", {"query": query})

@tool
async def mcp_metrics_db(metric_name: str) -> str:
    """Query benchmark metrics from the MCP metric database."""
    return await client.call_tool("mcp_metrics_db", {"metric_name": metric_name})

tools = [mcp_document_lookup, mcp_metrics_db]
print("Loaded MCP Tools:", [t.name for t in tools])

## 3. Invoking Agent with MCP Tools

In [ ]:
llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    temperature=0.1,
)

llm_with_tools = llm.bind_tools(tools)
response = await llm_with_tools.ainvoke("Check the 2-qubit gate fidelity metrics using the document lookup tool.")
print("LLM Tool Calls Generated:", response.tool_calls)

for call in response.tool_calls:
    if call["name"] == "mcp_document_lookup":
        res = await mcp_document_lookup.ainvoke(call["args"])
        print("\nMCP Tool Execution Result:", res)